# Processing NSIDC data

### Set up
#### Packages

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import datetime as dt
from datetime import timedelta
xr.set_options(display_expand_data=False);
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client
from dask.diagnostics import ProgressBar
from Processing_functions import AddCyclic, FixLongitude

#### Filepaths & name variables

In [2]:
## File name
filestem1 = 'N_'
filestem2 = '_extent_v4.0.csv'

## Filepaths
path_to_arch = "/glade/work/glydia/processed_NSIDC_data/"
comp = 'ice'
var_ind = 0

# Variables
var_list = {'ice': ['aice']}
var = var_list[comp][var_ind]

# Extensions
h_ext = {'ice': ['.h.']}

path_to_outdata = '/glade/work/glydia/processed_NSIDC_data/'

### Load & modify data
#### Control data

In [3]:
%%time

## Load data
# Open csvs
mon_range = np.arange(1,13)

processed_list = []
for i in range(0,len(mon_range)):
    mon = mon_range[i]
    mon_str = str(mon).zfill(2)
    df = pd.read_csv(path_to_arch+filestem1+mon_str+filestem2,na_values=-9999.00,skipinitialspace=True)

    # Create time array
    yr_array = df.year.values
    date_mon = pd.date_range(str(yr_array[0])+'-'+mon_str+'-01',str(yr_array[-1])+'-'+mon_str+'-01', freq='12MS')

    # Extract extent data
    extent_data = df.extent.values

    # Create data array
    ds = xr.DataArray(data=extent_data, coords={'time':date_mon}, name=var)
    processed_list.append(ds)
    
processed_out = xr.concat(processed_list,dim='time').sortby('time')

processed_out.to_netcdf(path_to_outdata+'NSIDC'+h_ext[comp][0]+var+'.195001-202412.'+'nc', 
                            format='NETCDF4',encoding={var: {"zlib": True, "complevel": 1}})
print('wrote data to disk')

wrote data to disk
CPU times: user 34.1 ms, sys: 17.1 ms, total: 51.3 ms
Wall time: 220 ms
